<a href="https://colab.research.google.com/github/kjs444642-alt/ml.practice1/blob/main/06_cross_validation_hyperparameter_tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 교차 검증과 그리드 서치

## 검증 데이터셋

In [ ]:
import pandas as pd

wine = pd.read_csv('https://bit.ly/wine-date')

### 문제 1 : wine 데이터 확인

In [ ]:
# wine 처음 5개 행 데이터 확인
wine.head()

,alcohol,sugar,pH,class
0,9.4,1.9,3.51,0.0
1,9.8,2.6,3.20,0.0
2,9.8,2.3,3.26,0.0
3,9.8,1.9,3.16,0.0
4,9.4,1.9,3.51,0.0


In [ ]:
print(wine.shape[0])

6497


In [ ]:
# wine 데이터 통계값 확인 (각 특성별 평균, 표준편차, 최소값, 최대값 등)
wine.describe()

,alcohol,sugar,pH,class
count,6497.000000,6497.000000,6497.000000,6497.000000
mean,10.491801,5.443235,3.218501,0.753886
std,1.192712,4.757804,0.160787,0.430779
min,8.000000,0.600000,2.720000,0.000000
25%,9.500000,1.800000,3.110000,1.000000
50%,10.300000,3.000000,3.210000,1.000000
75%,11.300000,8.100000,3.320000,1.000000
max,14.900000,65.800000,4.010000,1.000000


In [ ]:
# 화이트 와인, 레드 와인 데이터 개수 확인
wine['class'].value_counts()

,count
class,
1.0,4898
0.0,1599


### 데이터셋 분류

In [ ]:
data = wine[['alcohol', 'sugar', 'pH']].to_numpy()
target = wine['class'].to_numpy()

In [ ]:
from sklearn.model_selection import train_test_split

train_input, test_input, train_target, test_target = train_test_split(
    data, target, test_size=0.2, random_state=42)

In [ ]:
sub_input, val_input, sub_target, val_target = train_test_split(
    train_input, train_target, test_size=0.2, random_state=42)

In [ ]:
print(sub_input.shape, val_input.shape)

(4157, 3) (1040, 3)


In [ ]:
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier(random_state=42)
dt.fit(sub_input, sub_target)

print(dt.score(sub_input, sub_target))
print(dt.score(val_input, val_target))

0.9971133028626413
0.864423076923077


## 교차 검증

In [ ]:
from sklearn.model_selection import cross_validate

scores = cross_validate(dt, train_input, train_target)
print(scores)

{'fit_time': array([0.00916004, 0.00912118, 0.01048064, 0.00895691, 0.01030183]), 'score_time': array([0.00135398, 0.00130415, 0.00138116, 0.00145674, 0.00151515]), 'test_score': array([0.86923077, 0.84615385, 0.87680462, 0.84889317, 0.83541867])}


In [ ]:
import numpy as np

print(np.mean(scores['test_score']))

0.855300214703487


In [ ]:
from sklearn.model_selection import StratifiedKFold

scores = cross_validate(dt, train_input, train_target, cv=StratifiedKFold())
print(np.mean(scores['test_score']))

0.855300214703487


In [ ]:
splitter = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_validate(dt, train_input, train_target, cv=splitter)
print(np.mean(scores['test_score']))

0.8574181117533719


## 하이퍼파라미터 튜닝

In [ ]:
from sklearn.model_selection import GridSearchCV

params = {'min_impurity_decrease': [0.0001, 0.0002, 0.0003, 0.0004, 0.0005]}

In [ ]:
gs = GridSearchCV(DecisionTreeClassifier(random_state=42), params, n_jobs=-1)

In [ ]:
gs.fit(train_input, train_target)

GridSearchCV(estimator=DecisionTreeClassifier(random_state=42), n_jobs=-1,
             param_grid={'min_impurity_decrease': [0.0001, 0.0002, 0.0003,
                                                   0.0004, 0.0005]})

In [ ]:
dt = gs.best_estimator_
print(dt.score(train_input, train_target))

0.9615162593804117


In [ ]:
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV

# Re-define params as it might not be in the current scope
params = {'min_impurity_decrease': [0.0001, 0.0002, 0.0003, 0.0004, 0.0005]}

# Re-initialize and fit GridSearchCV
gs = GridSearchCV(DecisionTreeClassifier(random_state=42), params, n_jobs=-1)
gs.fit(train_input, train_target)

best_index = np.argmax(gs.cv_results_['mean_test_score'])
print(gs.cv_results_['params'][best_index])

{'min_impurity_decrease': 0.0001}


In [ ]:
print(gs.cv_results_['mean_test_score'])

[0.86819297 0.86453617 0.86492226 0.86780891 0.86761605]


In [ ]:
best_index = np.argmax(gs.cv_results_['mean_test_score'])
print(gs.cv_results_['params'][best_index])

{'min_impurity_decrease': 0.0001}


In [ ]:
params = {'min_impurity_decrease': np.arange(0.0001, 0.001, 0.0001),
          'max_depth': range(5, 20, 1),
          'min_samples_split': range(2, 100, 10)
          }

In [ ]:
gs = GridSearchCV(DecisionTreeClassifier(random_state=42), params, n_jobs=-1)
gs.fit(train_input, train_target)

GridSearchCV(estimator=DecisionTreeClassifier(random_state=42), n_jobs=-1,
             param_grid={'max_depth': range(5, 20),
                         'min_impurity_decrease': array([0.0001, 0.0002, 0.0003, 0.0004, 0.0005, 0.0006, 0.0007, 0.0008,
       0.0009]),
                         'min_samples_split': range(2, 100, 10)})

In [ ]:
print(gs.best_params_)

{'min_impurity_decrease': 0.0001}


In [ ]:
print(np.max(gs.cv_results_['mean_test_score']))

0.8681929740134745


In [ ]:
# 교차검증 수행 시간 프린트
gs.cv_results_['mean_fit_time']

array([0.01390052, 0.00976567, 0.00921783, 0.00814013, 0.00779347])

### 랜덤 서치

In [ ]:
from scipy.stats import uniform, randint

In [ ]:
# 균등 분포 샘플링
rgen = randint(0, 10)
rgen.rvs(10)

array([9, 7, 3, 6, 0, 7, 3, 5, 5, 9])

In [ ]:
np.unique(rgen.rvs(1000), return_counts=True) # 빈도도 함께 출력

(array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9]),
 array([107,  96,  90, 107, 100,  98, 102,  94,  93, 113]))

In [ ]:
ugen = uniform(0, 1)
ugen.rvs(10)

array([0.54040591, 0.49697754, 0.86542051, 0.99626631, 0.55898975,
       0.75177618, 0.45086796, 0.97752275, 0.04847949, 0.40607202])

In [ ]:
params = {'min_impurity_decrease': uniform(0.0001, 0.001),
          'max_depth': randint(20, 50),
          'min_samples_split': randint(2, 25),
          'min_samples_leaf': randint(1, 25),
          }

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

rs = RandomizedSearchCV(DecisionTreeClassifier(random_state=42), params,
                        n_iter=100, n_jobs=-1, random_state=42)
rs.fit(train_input, train_target)

RandomizedSearchCV(estimator=DecisionTreeClassifier(random_state=42),
                   n_iter=100, n_jobs=-1,
                   param_distributions={'max_depth': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x7cf0ff623440>,
                                        'min_impurity_decrease': <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x7cf0ff620ec0>,
                                        'min_samples_leaf': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x7cf0ffbe87d0>,
                                        'min_samples_split': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x7cf0ff621ca0>},
                   random_state=42)

In [ ]:
print(rs.best_params_)

{'max_depth': 39, 'min_impurity_decrease': np.float64(0.00034102546602601173), 'min_samples_leaf': 7, 'min_samples_split': 13}


In [ ]:
print(np.max(rs.cv_results_['mean_test_score']))

0.8695428296438884


In [ ]:
dt = rs.best_estimator_

print(dt.score(test_input, test_target))

0.86


In [ ]:
rs.cv_results_['mean_fit_time']

array([0.00780931, 0.00836544, 0.00871005, 0.00808425, 0.0074192 ,
       0.00874085, 0.00923166, 0.00781002, 0.01064367, 0.01139555,
       0.00956712, 0.0078198 , 0.00711107, 0.00760646, 0.00689831,
       0.00766134, 0.00697265, 0.00732145, 0.00923595, 0.00799093,
       0.00846162, 0.00703378, 0.0074533 , 0.00717802, 0.00882316,
       0.00831194, 0.00738297, 0.0075232 , 0.00748372, 0.0069159 ,
       0.00797462, 0.0085887 , 0.00699973, 0.00792961, 0.00813413,
       0.00796041, 0.00701914, 0.0093318 , 0.00708041, 0.00771518,
       0.00765133, 0.00785952, 0.00751314, 0.00920801, 0.00848927,
       0.00825243, 0.00684724, 0.00759702, 0.00807147, 0.00862069,
       0.00752473, 0.00945206, 0.00870275, 0.00884132, 0.00929561,
       0.00919614, 0.00766244, 0.00792284, 0.00717096, 0.00737238,
       0.00878057, 0.00792613, 0.00741906, 0.00751567, 0.00696878,
       0.00676174, 0.00680766, 0.00721159, 0.00834055, 0.00689602,
       0.00671148, 0.0088119 , 0.00706725, 0.00689983, 0.00879

In [ ]:
print(np.mean(rs.cv_results_['mean_fit_time']))

0.00893129825592041


### 결정트리 분할 옵션 변경

In [ ]:
rs2 = RandomizedSearchCV(DecisionTreeClassifier(splitter='random', random_state=42), params,
                        n_iter=100, n_jobs=-1, random_state=42)
rs2.fit(train_input, train_target)

RandomizedSearchCV(estimator=DecisionTreeClassifier(random_state=42,
                                                    splitter='random'),
                   n_iter=100, n_jobs=-1,
                   param_distributions={'max_depth': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x7cf0ff623440>,
                                        'min_impurity_decrease': <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x7cf0ff620ec0>,
                                        'min_samples_leaf': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x7cf0ffbe87d0>,
                                        'min_samples_split': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x7cf0ff621ca0>},
                   random_state=42)

In [ ]:
print(rs2.best_params_)
print(np.max(rs2.cv_results_['mean_test_score']))

dt = rs2.best_estimator_
print(dt.score(test_input, test_target))

{'max_depth': 43, 'min_impurity_decrease': np.float64(0.00011407982271508446), 'min_samples_leaf': 19, 'min_samples_split': 18}
0.8458726956392981
0.786923076923077


In [ ]:
rs2.cv_results_['mean_fit_time']

array([0.0070385 , 0.00874591, 0.00804129, 0.00846405, 0.0092814 ,
       0.01007481, 0.0072103 , 0.00703478, 0.00918303, 0.00647726,
       0.00760417, 0.00874462, 0.00963488, 0.00804462, 0.0069685 ,
       0.0077034 , 0.00754676, 0.00843577, 0.0084734 , 0.00742135,
       0.00659523, 0.00687246, 0.00482268, 0.0070538 , 0.00750589,
       0.00988593, 0.00688243, 0.00715547, 0.00830965, 0.00731902,
       0.00886149, 0.00908446, 0.00806355, 0.00765839, 0.00810394,
       0.00704546, 0.00594501, 0.00989408, 0.00770435, 0.00668983,
       0.00750036, 0.0078856 , 0.00729523, 0.00940347, 0.00932355,
       0.00472336, 0.00367227, 0.0066741 , 0.00949535, 0.00754271,
       0.0078474 , 0.00485864, 0.00645804, 0.00699091, 0.00820012,
       0.00799918, 0.00780129, 0.00876894, 0.00824771, 0.00839295,
       0.00561051, 0.00390787, 0.00322003, 0.00894728, 0.00743747,
       0.00675573, 0.00334044, 0.00662017, 0.00923004, 0.00897889,
       0.00499349, 0.00629315, 0.00484905, 0.00892334, 0.00538

In [ ]:
print(np.mean(rs2.cv_results_['mean_fit_time']))

0.007261259078979494


문제 2 : 위 코드가 기존 랜덤 서치 코드와 다른 점을 2가지 적어보세요.

결정트리 분할 방식 변경


무작위 분할로 인해 다양한 트리 구조 탐색 가능